In [ ]:
# Construct a simple parabola for a ballistic missile that launches with velocity v0
# at elevtion angle elevation0 and is only accelerated by gravity.
import numpy as np

from theia.coordinates import CoordinateTransformations, EcefToEnuTransformer
from theia.distance import line_of_sight_distance
from theia.terrain import SrtmTerrainModel
from theia.types import Point


terrain = SrtmTerrainModel()

start_lat = 54.7640
start_lon = 20.4080
start_alt = 30.0
p_start = Point(
    lat=start_lat,
    lon=start_lon,
    alt=start_alt,
)

stop_lat = 46.946667
stop_lon = 7.444167
stop_alt = terrain.elevationAt(46.946667, 7.444167)
p_stop = Point(
    lat=stop_lat,
    lon=stop_lon,
    alt=stop_alt,
)


# Earth curvature and trajectories for some launch angles along the great circle
# between the start and the stop point.
# Alpha is the elevation relative to the LOS vector between start and stop points!
alpha = np.deg2rad(30.0)


def build_trajectory(alpha, p_start, p_stop):
    distance = line_of_sight_distance(*p_start.as_tuple(), *p_stop.as_tuple())

    p_start_ecef = np.array(
        CoordinateTransformations.geodetic_to_cartesian(*p_start.as_tuple())
    )
    p_stop_ecef = np.array(
        CoordinateTransformations.geodetic_to_cartesian(*p_stop.as_tuple())
    )
    direction = p_stop_ecef - p_start_ecef
    direction /= np.linalg.norm(direction)

    v0 = np.sqrt(9.81 * distance / (2 * np.cos(alpha) * np.sin(alpha)))
    t_stop = 2 * np.sin(alpha) * v0 / 9.81

    xs = []
    ys = []
    for t in np.arange(0, t_stop, 10):
        x = np.cos(alpha) * v0 * t
        y = np.sin(alpha) * v0 * t - 0.5 * 9.81 * t**2
        xs.append(x)
        ys.append(y)

    return v0, t_stop, np.vstack([xs, ys]).T


def build_earth_curvature(p_start, p_stop):
    distance = line_of_sight_distance(*p_start.as_tuple(), *p_stop.as_tuple())

    p_start_ecef = np.array(
        CoordinateTransformations.geodetic_to_cartesian(*p_start.as_tuple())
    )
    p_stop_ecef = np.array(
        CoordinateTransformations.geodetic_to_cartesian(*p_stop.as_tuple())
    )
    direction = p_stop_ecef - p_start_ecef
    direction /= np.linalg.norm(direction)

    xs = list(np.arange(0, distance, 100))
    ys = []
    for d in xs:
        p_los_ecef = p_start_ecef + d * direction
        p_los_geodetic = CoordinateTransformations.cartesian_to_geodetic(*p_los_ecef)
        y = -p_los_geodetic[2]
        ys.append(y)

    return np.vstack([xs, ys]).T

In [ ]:
from matplotlib import pyplot as plt


fig, ax = plt.subplots()
data = build_earth_curvature(p_start, p_stop) / 1e3
ax.plot(data[:, 0], data[:, 1], "--", label="earth curvature")

for alpha in [20.0, 30.0, 40.0, 60.0]:
    v0, t_stop, data = build_trajectory(np.deg2rad(alpha), p_start, p_stop)
    data /= 1e3
    ax.plot(
        data[:, 0],
        data[:, 1],
        "--",
        label=rf"Trajectory for $\alpha = {alpha} \degree$; T = {t_stop / 60:.1f}min; $v_0$ = {v0:.0f} m/s",
    )

ax.set_xlabel("LOS distance [km]", fontsize=14)
ax.set_ylabel("Altitude above LOS [km]", fontsize=14)
ax.legend();

In [ ]:
from theia.data_loading import load_bakom_ukw_transmitters


txs = load_bakom_ukw_transmitters()
tx_positions = {}
for tx in txs:
    tx_positions[tx.point.as_tuple()] = max(
        tx.power, tx_positions.get(tx.point.as_tuple(), 0.0)
    )
# tx_positions = list(set(tx_positions.values()))
tx_positions = dict(sorted(list(tx_positions.items()), key=lambda item: item[1])[-10:])

In [ ]:
tx_positions

In [ ]:
import numpy as np

from theia.terrain import SrtmTerrainModel
from theia.test_data import build_pcl_receiver
from theia.types import Point, Receiver

terrain = SrtmTerrainModel()

rx_lat = 47.5661
rx_lon = 9.1079

p_rx = Point(
    lat=rx_lat,
    lon=rx_lon,
    alt=terrain.elevationAt(rx_lat, rx_lon),
)


rx = build_pcl_receiver(rx_id=0, point=p_rx)

In [ ]:
tx_santis = next(tx for tx in txs if tx.id == 890)

In [ ]:
from theia.config import SIDC
from theia.detection.pcl import PclDetector
from theia.grids import LatLonHeightGrid
from theia.types import PclMeasurementModel, PclSensor, Target
from theia.util import mask_to_polygon


grid = LatLonHeightGrid(
    lat_start=47.1334,
    lat_stop=47.7983,
    lat_res=0.01,
    lon_start=8.4214,
    lon_stop=9.8619,
    lon_res=0.01,
    height_start=1000.0,
    height_stop=1000.0,
    height_res=100.0,
)

detector = PclDetector()
min_rcs = detector.minimum_detectable_rcs_grid(rx, tx_santis, grid)
polygon = mask_to_polygon(
    (min_rcs[:, :, 0] <= 1.0),
    grid.latitude_values[0],
    grid.latitude_values[1] - grid.latitude_values[0],
    grid.longitude_values[0],
    grid.longitude_values[1] - grid.longitude_values[0],
)

In [ ]:
polygon[1]

In [ ]:
from matplotlib import pyplot as plt

fig, ax = plt.subplots()

img = ax.imshow(min_rcs[:, :, 0] <= 1.0)
fig.colorbar(img)

In [ ]:
import folium

map = folium.Map(location=(47.46083, 8.55528))
# folium.GeoJson(traj.to_geojson()).add_to(map)
for pos, power in tx_positions.items():
    folium.Marker(
        (pos[0], pos[1]),
        tooltip=f"Lat={pos[0]}, Lon={pos[1]} | {power:.0f} W",
    ).add_to(map)
folium.Marker((rx_lat, rx_lon), color="red").add_to(map)
for p in polygon:
    folium.GeoJson(p).add_to(map)
folium.LatLngPopup().add_to(map)
map

In [ ]:
import datetime

import numpy as np

from theia.simulation.factories.zurich_airport import ZurichAirportScenarioFactory
from theia.terrain import SrtmTerrainModel

terrain = SrtmTerrainModel()

factory = ZurichAirportScenarioFactory(
    np.random.default_rng(seed=30557092),
    terrain_model=terrain,
)
traj = factory._build_single_drone(
    0.0, datetime.datetime.fromtimestamp(0)
)._child.trajectory
target = traj(datetime.datetime.fromtimestamp(0))
target.point

In [ ]:
t = traj.times[0]
t

In [ ]:
factory._t0 < t

In [ ]:
from theia.simulation.theia_logging import LogLoader


log = LogLoader("log.json")

In [ ]:
[d for d in log.blue_monostatic_radar_detections if d.target.id == 46]

In [ ]:
import datetime

s = [
    s
    for s in log._snapshots
    if s.time
    == datetime.datetime(
        year=2026,
        month=1,
        day=1,
        hour=0,
        minute=7,
        second=7,
        tzinfo=datetime.timezone.utc,
    )
]
assert len(s) == 1
s = s[0]
target = [t for t in s.red_targets if t.id == 46]
assert len(target) == 1
target = target[0]
radar = log.blue_monostatic_radars[1]

In [ ]:
import numpy as np

from theia.detection.active import calculate_monostatic_detection
from theia.terrain import SrtmTerrainModel

# TODO: Something with the terrain!!!

calculate_monostatic_detection(
    SrtmTerrainModel(), radar, target, np.random.default_rng(seed=3905789)
)

In [ ]:
from theia.config import SIDC
from theia.types import ConstantRcsModel, Point, Target

p_target = Point(lat=47.6659, lon=8.6569, alt=1000.0)

In [ ]:
from theia.terrain import SrtmTerrainModel
from theia.test_data import build_flores_monostatic_radar
from theia.types import Point

terrain_model = SrtmTerrainModel()

p_weissfluh = Point(
    lat=46.834503,
    lon=9.795055,
    alt=terrain_model.elevationAt(46.834503, 9.795055),
)

radar_weissfluh = build_flores_monostatic_radar(
    p_weissfluh,
    sensor_id=0,
    rx_id=0,
    tx_id=0,
)

In [ ]:
p = Point(lat=47.7808, lon=)

In [ ]:
from theia.coverage import calculate_coverage


p_flak_klosters = Point(
    lat=46.87991,
    lon=9.87260,
    alt=terrain_model.elevationAt(46.87991, 9.87260),
)

cov = calculate_coverage(terrain_model, p_flak_klosters, 4_000, 2000)
type(cov)

In [ ]:
from theia.types import GeoJSONFeature, GeoJSONPolygon


GeoJSONFeature(
    geometry=GeoJSONPolygon.from_shapely(cov),
    properties={"name": "my polygon"},
)

In [ ]:
import itertools


list(itertools.chain.from_iterable([["a"], [], []]))

In [ ]:
from theia.coverage import calculate_coverage
from theia.radar_equation import calculate_maximum_monostatic_range

d_max = calculate_maximum_monostatic_range(radar_weissfluh, 1.0)

calculate_coverage(terrain_model, p_weissfluh, d_max, 1600)

In [ ]:
import numpy as np

from theia.simulation.factories.davos_drone import DavosDroneScenarioFactory
from theia.terrain import SrtmTerrainModel


factory = DavosDroneScenarioFactory(np.random.default_rng(), SrtmTerrainModel())
c = factory._get_red_controller()
kamikaze = c._controllers[0]
traj = kamikaze.trajectory

In [ ]:
factory._get_blue_controller()._target_id

In [ ]:
p = traj(traj.times[-1]).point

In [ ]:
SrtmTerrainModel().has_line_of_sight(factory._p_infra, p)

In [ ]:
import folium


map = folium.Map()
folium.Marker((p.lat, p.lon)).add_to(map)
folium.Marker(
    (factory._p_infra.lat, factory._p_infra.lon), icon=folium.Icon(color="red")
).add_to(map)
map

In [ ]:
import datetime

from theia.config import SIDC
from theia.types import SituationalPicture, Track

t = datetime.datetime(year=2026, month=1, day=1, second=4)

pic = SituationalPicture(
    time=t,
    friendly_radars=[],
    friendly_pet_receivers=[],
    friendly_targets=[],
    enemy_targets=[
        Track(
            id="0",
            sidc=SIDC.RED_FIXED_WING,
            states=[
                (
                    t,
                    np.array(
                        [
                            4310662.8130001165,
                            0.0,
                            747001.7049874394,
                            0.0,
                            4627767.526008632,
                            0.0,
                        ]
                    ),
                )
            ],
        )
    ],
)

c.get_firing_effectors(pic, datetime.timedelta(seconds=1))

In [ ]:
from theia.coordinates import CoordinateTransformations


CoordinateTransformations.geodetic_to_cartesian(
    46.800885331290964, 9.83124318228007, 1546.1545442268252
)

In [ ]:
from theia.types import SituationalPicture


SituationalPicture.model_validate_json("""
    {
      "id": "0",
      "points": [
        {
          "time": "2026-01-01T00:00:04",
          "lat": 46.800885331290964,
          "lon": 9.83124318228007,
          "alt": 1546.1545442268252,
          "v_east": 0,
          "v_north": 0,
          "v_up": 0
        }
      ],
      "sidc": "10232000001206000000",
      "receiver": null,
      "transmitter": null
    }""")

In [ ]:
# TODO
# Ballistic missile, e. g. PrSM


# Cruise Missile
# Iskander M
# https://handwiki.org/wiki/Engineering:9K720_Iskander

In [ ]:
# import numpy as np
# from scipy.constants import gravitational_constant

# omega = 0.0
# """Earth rotation rate"""
# G = gravitational_constant
# M = 5.972 * 1e24
# """Mass of earth [kg]"""
# r0 = 1.0
# mu = 0.0
# """Geodetic latitude of radar [°]"""
# rho = 0.0
# g = 9.81
# beta = 1.0
# V0 = 1.0
# R = 1.0
# V = 1.0
# r = 1.0
# J = 1.0
# a = 1.0
# phi = 1.0
# mu_c = 1.0
# """Geocentric latitude of radar [°]"""

# # State vector:
# # x, y, z, vx, vy, vz
# x = np.ones(6)


# A_top_left = np.zeros((3, 3))
# A_top_right = np.eye(3)
# # fmt: off
# tmp1 = -omega ** 2 * np.sin(mu) * np.cos(mu)
# tmp2 = - rho * g / (2 * beta) * V0

# A_bottom_left = np.array([
#     [omega**2 - G * M / r0**3,                                      0.0, 0.0],
#     [                     0.0, omega**2 * np.sin(mu)**2 - G * M / r0**3,                                      tmp1],
#     [                     0.0,                                      tmp1, omega**2 * np.cos(mu)**2 - G * M / r0**3],
# ])

# A_bottom_right = np.array([
#     [                   tmp2, 2 * omega * np.sin(mu), -2 * omega * np.cos(mu)],
#     [-2 * omega * np.sin(mu),                   tmp2,                     0.0],
#     [ 2 * omega * np.cos(mu),                    0.0,                    tmp2],
# ])

# A = np.block(
#     [
#         [A_top_left, A_top_right],
#         [A_bottom_left, A_bottom_right],
#     ]
# )

# B = np.array(
#     [
#         0.0,
#         0.0,
#         0.0,
#         0.0,
#         -(omega**2) * R * np.sin(mu) * np.cos(mu),
#         omega**2 * R * np.cos(mu) ** 2 - G * M * R / r0**3,
#     ]
# )

# C = np.array(
#     [
#         0.0,
#         0.0,
#         0.0,
#         rho * g / (2 * beta) * (V0 - V) * x[3] + G * M * (1 / r0**3 - 1 / r**3) * x[0],
#         rho * g / (2 * beta) * (V0 - V) * x[4] + G * M * (1 / r0**3 - 1 / r**3) * x[1],
#         rho * g / (2 * beta) * (V0 - V) * x[5] + G * M * (1 / r0**3 - 1 / r**3) * (x[2] + R),
#     ]
# )

# D = np.array(
#     [
#         0.0,
#         0.0,
#         0.0,
#         - G * M / r**3 * J * (a / r)**2 * (1.0 - 5 * np.sin(phi) ** 2) * x[0],
#         - G * M / r**3 * J * (a / r)**2 * (1.0 - 5 * np.sin(phi) ** 2) * x[1] - 2 * G * M / r**2 * J * (a / r)**2 * np.sin(phi) * np.cos(mu) \
#             - R * omega**2 * np.sin(mu) * (np.cos(mu) * (np.cos(mu - mu_c) - 1) - np.sin(mu) * np.sin(mu - mu_c)) \
#             + G * M / r**3 * R * np.sin(mu - mu_c) * (J * (a / r)**2 * (1.0 - 5 * np.sin(phi)**2) + 1.0),
#         - G * M / r**3 * J * (a / r)**2 * (1.0 - 5 * np.sin(phi) ** 2) * (x[2] + R * np.cos(mu - mu_c)) - 2 * G * M / r**2 * J * (a / r)**2 * np.sin(phi) * np.cos(mu) \
#             + R * omega**2 * np.cos(mu) * (np.cos(mu) * (np.cos(mu - mu_c) - 1) + np.sin(mu) * np.sin(mu - mu_c)) \
#             - G * M / r**3 * R * (np.cos(mu - mu_c) - 1),
#     ]
# )
# # fmt: on

In [ ]:
from theia.terrain import SrtmTerrainModel


terrain = SrtmTerrainModel()

In [ ]:
from theia.types import Point


THEATER_LAT_MIN = 46.3805
THEATER_LON_MIN = 9.2207
THEATER_LAT_MAX = 47.2440
THEATER_LON_MAX = 10.4790

# Source:
# AIRAC AIP SUP: 008/2025
# https://www.skybriefing.com/documents/10156/531923/LS_Sup_A_2025_008_en.pdf/9383f427-aee4-73e5-2f58-f6c49c721bc8?t=1766406342056
# Retrieved 2026-07-02.

NM_TO_M = 1852
FEET_TO_M = 0.3048

DAVOS_LAT = 46.81472
DAVOS_LON = 9.84944
RADIUS_RESTRICTED_AREA = 25 * NM_TO_M  # [m]
RADIUS_DAVOS_CONTROL_ZONE = 2.7 * NM_TO_M  # [m]
HEIGHT_RESTRICTED_AREA = 19500 * FEET_TO_M  # [m]

davos = Point(
    lat=DAVOS_LAT,
    lon=DAVOS_LON,
    alt=terrain.elevationAt(DAVOS_LAT, DAVOS_LON),
)

In [ ]:
import json
import folium

data_dir = "/home/user/Documents/theia_backend/examples/conference/data"

with open(f"{data_dir}/LS-R90.geojson", "r") as file:
    lsr90 = json.load(file)

with open(f"{data_dir}/flight-path-spacetime_authorized.json", "r") as file:
    data_authorized = json.load(file)

with open(f"{data_dir}/flight-path-spacetime_adversarial.json", "r") as file:
    data_adversarial = json.load(file)


flight_path_green = folium.PolyLine(
    [(p["lat"], p["lon"]) for p in data_authorized if p["time"] <= 1550],
    tooltip="Normal flight",
    color="green",
)
flight_path_green_planned = folium.PolyLine(
    [(p["lat"], p["lon"]) for p in data_authorized if p["time"] > 1550],
    tooltip="Normal flight (authorized)",
    color="green",
    dashArray="5, 5",
)
flight_path_red = folium.PolyLine(
    [(p["lat"], p["lon"]) for p in data_adversarial if p["time"] > 1550],
    tooltip="Unauthorized change of plan: RED!",
    color="red",
)

In [ ]:
import folium


highlight_waypoints = [
    "AKABI",
    "BODAN",
    "LAGOS",
    "VEBEG",
    "EBUXA",
    "ARGAX",
]


map = folium.Map(location=(DAVOS_LAT, DAVOS_LON), zoom_start=9)
folium.LatLngPopup().add_to(map)
folium.Rectangle(
    [
        (THEATER_LAT_MIN, THEATER_LON_MIN),
        (THEATER_LAT_MAX, THEATER_LON_MAX),
    ],
    tooltip="Theater",
    color="black",
).add_to(map)

folium.GeoJson(
    lsr90,
    tooltip="TEMPO LS-R90",
    fill=False,
    color="orange",
).add_to(map)
folium.Circle(
    location=(DAVOS_LAT, DAVOS_LON),
    radius=RADIUS_DAVOS_CONTROL_ZONE,
    tooltip="CTR",
    color="darkred",
).add_to(map)

folium.Marker(
    flight_path_green.locations[-1],
    tooltip="Transponder off",
    icon=folium.Icon(icon="power-off", prefix="fa", color="red"),
).add_to(map)

# folium.Marker((points[:, :, :, 0].min(), points[:, :, :, 1].min())).add_to(map)
# folium.Marker((points[:, :, :, 0].max(), points[:, :, :, 1].max())).add_to(map)
flight_path_green.add_to(map)
flight_path_green_planned.add_to(map)
flight_path_red.add_to(map)

map

In [ ]:
from theia.coordinates import CoordinateTransformations, EcefToEnuTransformer
from theia.grids import EnuGridMask

transformer = EcefToEnuTransformer(reference_point=davos)
p_northeast_enu = transformer.ecef_to_enu(
    CoordinateTransformations.geodetic_to_cartesian(
        THEATER_LAT_MAX,
        THEATER_LON_MAX,
        0.0,
    )
)
p_southwest_enu = transformer.ecef_to_enu(
    CoordinateTransformations.geodetic_to_cartesian(
        THEATER_LAT_MIN,
        THEATER_LON_MIN,
        0.0,
    )
)

davos_enu = transformer.ecef_to_enu(
    CoordinateTransformations.geodetic_to_cartesian(
        davos.lat,
        davos.lon,
        davos.alt,
    )
)

RES_NORTH_EAST = 100.0
RES_UP = 100.0

grid = EnuGridMask(
    reference_point=davos,
    east_min=p_southwest_enu[0],
    east_max=p_northeast_enu[0],
    north_min=p_southwest_enu[1],
    north_max=p_northeast_enu[1],
    up_min=davos_enu[2],
    up_max=davos_enu[2] + 1500,
    res_north_east=RES_NORTH_EAST,
    res_up=RES_UP,
    terrain=terrain,
)
mask = grid.binary_mask
points = grid.points

In [ ]:
from matplotlib import pyplot as plt
import numpy as np


fig, axes = plt.subplots(3, 3, figsize=(8, 9))

for i, ax in enumerate(axes.flatten()):
    ax.imshow((mask[:, :, i]))
    ax.set_title(f"Up = {i * RES_UP}m")
    ax.invert_yaxis()
    ax.set_xlabel(f"East [{RES_NORTH_EAST} m]")
    ax.set_ylabel(f"North [{RES_NORTH_EAST} m]")

fig.tight_layout()

In [ ]:
import numba


@numba.njit(cache=True)
def _bfs_shortest_path(occ, start, stop, connectivity):
    nx, ny, nz = occ.shape
    n = nx * ny * nz

    visited = np.zeros(n, dtype=np.bool_)
    prev = np.full(n, -1, dtype=np.int64)
    queue = np.empty(n, dtype=np.int64)
    q_head = 0
    q_tail = 0

    sx, sy, sz = start[0], start[1], start[2]
    ex, ey, ez = stop[0], stop[1], stop[2]

    if occ[sx, sy, sz] or occ[ex, ey, ez]:
        return np.empty(0, dtype=np.int64), False

    s_idx = (sx * ny + sy) * nz + sz
    e_idx = (ex * ny + ey) * nz + ez

    visited[s_idx] = True
    queue[q_tail] = s_idx
    q_tail += 1

    if connectivity == 6:
        dxs = np.array([1, -1, 0, 0, 0, 0], dtype=np.int64)
        dys = np.array([0, 0, 1, -1, 0, 0], dtype=np.int64)
        dzs = np.array([0, 0, 0, 0, 1, -1], dtype=np.int64)
    else:  # 26-connectivity
        dxs = np.empty(26, dtype=np.int64)
        dys = np.empty(26, dtype=np.int64)
        dzs = np.empty(26, dtype=np.int64)
        c = 0
        for dx in (-1, 0, 1):
            for dy in (-1, 0, 1):
                for dz in (-1, 0, 1):
                    if dx == 0 and dy == 0 and dz == 0:
                        continue
                    dxs[c] = dx
                    dys[c] = dy
                    dzs[c] = dz
                    c += 1

    found = False
    while q_head < q_tail:
        cur = queue[q_head]
        q_head += 1

        if cur == e_idx:
            found = True
            break

        cz = cur % nz
        cy = (cur // nz) % ny
        cx = cur // (ny * nz)

        for k in range(dxs.shape[0]):
            nxp = cx + dxs[k]
            nyp = cy + dys[k]
            nzp = cz + dzs[k]

            if 0 <= nxp < nx and 0 <= nyp < ny and 0 <= nzp < nz:
                nidx = (nxp * ny + nyp) * nz + nzp
                if (not visited[nidx]) and (not occ[nxp, nyp, nzp]):
                    visited[nidx] = True
                    prev[nidx] = cur
                    queue[q_tail] = nidx
                    q_tail += 1

    if not found:
        return np.empty(0, dtype=np.int64), False

    # reconstruct path length first, then fill backward
    path_len = 1
    node = e_idx
    while node != s_idx:
        node = prev[node]
        path_len += 1

    path = np.empty(path_len, dtype=np.int64)
    node = e_idx
    for i in range(path_len - 1, -1, -1):
        path[i] = node
        if node != s_idx:
            node = prev[node]

    return path, True


def shortest_path(occ, start, stop, connectivity=6):
    """
    Shortest path between two voxels in a 3D binary occupancy grid (BFS).

    Parameters
    ----------
    occ : np.ndarray[bool], shape (nx, ny, nz)
        True = occupied/blocked, False = free.
    start, stop : tuple of int (x, y, z)
        Voxel coordinates.
    connectivity : int, 6 or 26
        6 = face neighbors only, 26 = face+edge+corner neighbors.

    Returns
    -------
    list of (x, y, z) tuples from start to stop (inclusive), or None if
    no path exists.
    """
    occ = np.ascontiguousarray(occ, dtype=np.bool_)
    nx, ny, nz = occ.shape

    start_arr = np.asarray(start, dtype=np.int64)
    stop_arr = np.asarray(stop, dtype=np.int64)

    flat_path, found = _bfs_shortest_path(occ, start_arr, stop_arr, connectivity)

    if not found:
        return None

    coords = []
    for idx in flat_path:
        z = idx % nz
        y = (idx // nz) % ny
        x = idx // (ny * nz)
        coords.append((int(x), int(y), int(z)))
    return coords


In [ ]:
%%timeit
path = shortest_path(mask, (850, 250, 0), (500, 100, 0))

In [ ]:
from theia.export_paraview import ParaviewExporter


ParaviewExporter(
    "paraview",
    THEATER_LAT_MIN,
    THEATER_LAT_MAX,
    0.001,
    THEATER_LON_MIN,
    THEATER_LON_MAX,
    0.001,
    terrain,
    elevation_factor=1.0,
).export([], [])

In [ ]:
points[:, :, 0, 2]

In [ ]:
map.save("map.html")

In [ ]:
from theia.types import Point
from theia.distance import line_of_sight_distance

p1 = Point(lat=47.36573878355709, lon=8.557384378716328, alt=800.27486575488)
p2 = Point(lat=47.367007613363995, lon=8.559369945700368, alt=825.1246458636597)

line_of_sight_distance(*p1.as_tuple(), *p2.as_tuple())